In [1]:
# =============================================================================
# STEP 6b - CORRECTED COMPARISON OF AGGREGATE AND CLASS-CONDITIONAL SHIFT
#
# Notebook 42 compared the two measures with a POOLED rank correlation and
# concluded the aggregate did better (0.840 against 0.789). That comparison is
# invalid and the conclusion is withdrawn.
#
# The aggregate takes exactly three distinct values across seventeen class cells,
# one per dataset, and is CONSTANT within each dataset. A pooled correlation
# therefore scores it on between-dataset severity ordering only. It is the same
# identification failure that sank the preregistered pooled beta5 model: a
# predictor that varies only between clusters cannot be compared, on pooled data,
# with one that varies within them.
#
# This notebook redoes the comparison within datasets, where the aggregate is
# constant by construction and therefore explains nothing at all. It reads the
# committed artefacts from notebook 42, so it costs seconds rather than re-fitting
# a hundred domain classifiers.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import rankdata
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
M=pd.read_csv(RD/'class_conditional_scov_vs_coverage.csv')
print('class cells:', len(M), '| datasets:', sorted(M.dataset.unique()))


Mounted at /content/drive
class cells: 17 | datasets: ['ciciot2023', 'nslkdd', 'ugr16']


In [2]:
# =============================================================================
# Cell 2 - establish the defect in the pooled comparison, numerically.
# =============================================================================
print('THE AGGREGATE IS CONSTANT WITHIN EACH DATASET')
agg=M.groupby('dataset')['S_cov_aggregate'].agg(['first','nunique'])
print(agg.rename(columns={'first':'value','nunique':'distinct values within dataset'}).to_string())
print(f'\n  {M.S_cov_aggregate.nunique()} distinct aggregate values across {len(M)} class cells.')
print('  Within a dataset it has zero variance, so it can order nothing there.')
print('  Its pooled correlation is entirely between-dataset information.')
r_pool_agg,_=stats.spearmanr(M.S_cov_aggregate, M.undercoverage)
r_pool_cls,_=stats.spearmanr(M.S_cov_class,     M.undercoverage)
print(f'\n  pooled: aggregate {r_pool_agg:+.3f} vs class-conditional {r_pool_cls:+.3f}')
print('  Notebook 42 read this as the aggregate winning. It is not a valid comparison.')


THE AGGREGATE IS CONSTANT WITHIN EACH DATASET
               value  distinct values within dataset
dataset                                             
ciciot2023  0.512268                               1
nslkdd      0.893399                               1
ugr16       0.712944                               1

  3 distinct aggregate values across 17 class cells.
  Within a dataset it has zero variance, so it can order nothing there.
  Its pooled correlation is entirely between-dataset information.

  pooled: aggregate +0.840 vs class-conditional +0.789
  Notebook 42 read this as the aggregate winning. It is not a valid comparison.


In [3]:
# =============================================================================
# Cell 3 - the correct comparison: within each dataset.
# =============================================================================
print('WITHIN-DATASET RANK CORRELATION OF S_cov,c WITH UNDERCOVERAGE')
res=[]
for ds,g in M.groupby('dataset'):
    spread=float(g.undercoverage.max()-g.undercoverage.min())
    if len(g)<3: continue
    r,p=stats.spearmanr(g.S_cov_class, g.undercoverage)
    res.append({'dataset':ds,'n':len(g),'rho':round(float(r),3),'p':round(float(p),4),
                'undercoverage_spread':round(spread,4)})
R=pd.DataFrame(res)
R['aggregate_rho']='undefined (constant)'
print(R.to_string(index=False))
print('\n  The aggregate has no within-dataset correlation at all: it is one number per dataset.')
print('  So class-conditional shift is strictly more informative within an environment,')
print('  and that is true by construction, not by contest.')

print('\nWHERE THE OUTCOME ACTUALLY VARIES')
varying=R[R.undercoverage_spread>0.05]
flat=R[R.undercoverage_spread<=0.05]
print(f'  environments with real variation in undercoverage: {list(varying.dataset)}')
for _,r in varying.iterrows():
    print(f'     {r.dataset:12s} rho={r.rho:+.3f} (n={r.n}, spread {r.undercoverage_spread:.3f})')
print(f'  environments with no failure to explain: {list(flat.dataset)}')
for _,r in flat.iterrows():
    print(f'     {r.dataset:12s} rho={r.rho:+.3f} (n={r.n}, spread {r.undercoverage_spread:.4f})'
          f'  <- range restriction, ordering noise')


WITHIN-DATASET RANK CORRELATION OF S_cov,c WITH UNDERCOVERAGE
   dataset  n    rho      p  undercoverage_spread        aggregate_rho
ciciot2023  8 -0.214 0.6103                0.0061 undefined (constant)
    nslkdd  4  0.800 0.2000                0.8970 undefined (constant)
     ugr16  5  0.900 0.0374                0.4153 undefined (constant)

  The aggregate has no within-dataset correlation at all: it is one number per dataset.
  So class-conditional shift is strictly more informative within an environment,
  and that is true by construction, not by contest.

WHERE THE OUTCOME ACTUALLY VARIES
  environments with real variation in undercoverage: ['nslkdd', 'ugr16']
     nslkdd       rho=+0.800 (n=4, spread 0.897)
     ugr16        rho=+0.900 (n=5, spread 0.415)
  environments with no failure to explain: ['ciciot2023']
     ciciot2023   rho=-0.214 (n=8, spread 0.0061)  <- range restriction, ordering noise


In [4]:
# =============================================================================
# Cell 4 - stratified test and the honest summary.
# =============================================================================
z=[]
for ds,g in M.groupby('dataset'):
    if len(g)<3: continue
    z.append(pd.DataFrame({'dataset':ds,'rs':rankdata(g.S_cov_class),
                           'ru':rankdata(g.undercoverage)}))
Z=pd.concat(z, ignore_index=True)
r_all,p_all=stats.spearmanr(Z.rs, Z.ru)
Zv=Z[Z.dataset.isin(varying.dataset)]
r_var,p_var=stats.spearmanr(Zv.rs, Zv.ru)
print('STRATIFIED (within-dataset ranks pooled)')
print(f'  all three environments        rho={r_all:+.3f} p={p_all:.4f} n={len(Z)}')
print(f'  environments that fail only   rho={r_var:+.3f} p={p_var:.4f} n={len(Zv)}')
print('\n  The all-environment figure is diluted by the environment where nothing fails:')
print('  eight of seventeen cells lie in CIC-IoT-2023, where every class is within 0.006')
print('  of nominal and there is no ordering to recover. This is the same range-restriction')
print('  effect already documented for the mechanism correlation.')

print('\nPER-DATASET DETAIL, ranked by class-conditional shift')
for ds,g in M.groupby('dataset'):
    g=g.sort_values('S_cov_class',ascending=False)
    print(f'\n  {ds} (aggregate {g.S_cov_aggregate.iloc[0]:.4f} for every class):')
    for _,r in g.iterrows():
        flag=''
        print(f'     {r["class"]:12s} S_cov,c {r.S_cov_class:.4f}  null {r.S_cov_null:.3f}  '
              f'coverage {r.coverage:.4f}{flag}')

print('\nVERDICT (supersedes notebook 42):')
print('  The aggregate cannot order class-level outcomes within an environment because it')
print('  is constant there. The class-conditional measure orders them well wherever the')
print(f'  outcome varies: rho {float(varying[varying.dataset=="ugr16"].rho.iloc[0]):+.3f} on UGR\u201916 and '
      f'{float(varying[varying.dataset=="nslkdd"].rho.iloc[0]):+.3f} on NSL-KDD.')
print('  On CIC-IoT-2023 it is uninformative because nothing fails, which is the correct')
print('  behaviour for a predictor of failure in an environment without failures.')
print('  The reviewer\u2019s objection is upheld: the study should not use an aggregate statistic')
print('  to explain class-conditional outcomes.')


STRATIFIED (within-dataset ranks pooled)
  all three environments        rho=+0.286 p=0.2658 n=17
  environments that fail only   rho=+0.862 p=0.0028 n=9

  The all-environment figure is diluted by the environment where nothing fails:
  eight of seventeen cells lie in CIC-IoT-2023, where every class is within 0.006
  of nominal and there is no ordering to recover. This is the same range-restriction
  effect already documented for the mechanism correlation.

PER-DATASET DETAIL, ranked by class-conditional shift

  ciciot2023 (aggregate 0.5123 for every class):
     Web          S_cov,c 0.6536  null 0.498  coverage 0.9519
     BruteForce   S_cov,c 0.5098  null 0.503  coverage 0.9558
     Recon        S_cov,c 0.5025  null 0.499  coverage 0.9497
     DoS          S_cov,c 0.4994  null 0.499  coverage 0.9499
     Spoofing     S_cov,c 0.4984  null 0.501  coverage 0.9528
     Mirai        S_cov,c 0.4958  null 0.502  coverage 0.9504
     DDoS         S_cov,c 0.4949  null 0.500  coverage 0.9500


In [5]:
# =============================================================================
# Cell 5 - save, supersede notebook 42's verdict, commit.
# =============================================================================
R.to_csv(RD/'class_conditional_scov_within_dataset.csv', index=False)
(RD/'class_conditional_scov_verdict.json').write_text(json.dumps({
 'supersedes':'the pooled comparison in notebook 42, which is withdrawn',
 'why_withdrawn':'the aggregate S_cov takes one value per dataset and is constant within a '
                 'dataset, so a pooled rank correlation scores it on between-dataset variation '
                 'alone; the same identification failure as the preregistered pooled model',
 'within_dataset_rho':R.to_dict('records'),
 'stratified_all':{'rho':float(r_all),'p':float(p_all),'n':int(len(Z))},
 'stratified_failing_environments':{'rho':float(r_var),'p':float(p_var),'n':int(len(Zv))},
 'aggregate_within_dataset':'undefined, zero variance',
 'conclusion':'aggregate shift statistics cannot explain class-conditional outcomes; the '
              'class-conditional measure orders them wherever the outcome varies'},
 indent=2, default=str))
print('saved within-dataset table and replaced the step 6 verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 6b: corrected within-dataset comparison of aggregate vs class-conditional shift; withdraws the pooled comparison in nb42')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved within-dataset table and replaced the step 6 verdict
nothing to commit

